In [ ]:
import biogeme.biogeme_logging as blog
import biogeme.biogeme as bio
from biogeme.expressions import Beta
from biogeme import models
from biogeme.database import Database
from biogeme.expressions import Beta, Variable
import pandas as pd

In [ ]:
df = pd.read_excel('RVM_data_project_clean.xlsx')
database = Database('RVM_data_project_clean.xlsx', df)

In [ ]:
df

In [ ]:
df['Age_10'] = df['Age'] / 10


First model - basically almost every variable used and not grouped. First model was to get the hang of it

In [ ]:
choice_code = Variable('choice_code')

Age_10 = Variable('Age')

Female = Variable('Female')

socioeconomic_level = Variable('socioeconomic_level')

dist = {i: Variable(f'dist_{i}') for i in range(1, 36)}

indoor = {i: Variable(f'indoor_{i}') for i in range(1, 36)}

university = {i: Variable(f'university_{i}') for i in range(1, 36)}

public_building = {i: Variable(f'public_building_{i}') for i in range(1, 36)}

private_building = {i: Variable(f'private_building_{i}') for i in range(1, 36)}

public_park = {i: Variable(f'public_park_{i}') for i in range(1, 36)}

hospital = {i: Variable(f'hospital_{i}') for i in range(1, 36)}

technical_institution = {i: Variable(f'technical_institution_{i}') for i in range(1, 36)}

science_park = {i: Variable(f'science_park_{i}') for i in range(1, 36)}

recreational_park = {i: Variable(f'recreational_park_{i}') for i in range(1, 36)}

downtown_recycling = {i: Variable(f'downtown_recycling_{i}') for i in range(1, 36)}

conv_center = {i: Variable(f'conv_center_{i}') for i in range(1, 36)}

shopping_center = {i: Variable(f'shopping_center_{i}') for i in range(1, 36)}

industrial_business_center = {i: Variable(f'industrial_business_center_{i}') for i in range(1, 36)}

In [ ]:
B_DIST = Beta('B_DIST', 0, None, None, 0)
B_INDOOR = Beta('B_INDOOR', 0, None, None, 0)
B_UNIVERSITY = Beta('B_UNIVERSITY', 0, None, None, 0)
B_PUBLIC_BUILDING = Beta('B_PUBLIC_BUILDING', 0, None, None, 0)
B_PRIVATE_BUILDING = Beta('B_PRIVATE_BUILDING', 0, None, None, 0)
B_PUBLIC_PARK = Beta('B_PUBLIC_PARK', 0, None, None, 0)
B_HOSPITAL = Beta('B_HOSPITAL', 0, None, None, 0)
B_TECHNICAL_INSTITUTION = Beta('B_TECHNICAL_INSTITUTION', 0, None, None, 0)
B_SCIENCE_PARK = Beta('B_SCIENCE_PARK', 0, None, None, 0)
B_RECREATIONAL_PARK = Beta('B_RECREATIONAL_PARK', 0, None, None, 0)
B_DOWNTOWN_RECYCLING = Beta('B_DOWNTOWN_RECYCLING', 0, None, None, 0)
B_CONV_CENTER = Beta('B_CONV_CENTER', 0, None, None, 0)
B_INDUSTRIAL_BUSINESS_CENTER = Beta('B_INDUSTRIAL_BUSINESS_CENTER', 0, None, None, 0)

In [ ]:
V = {}

for i in range(1, 36):
    V[i] = (
        B_DIST * dist[i]
        + B_INDOOR * indoor[i]
        + B_UNIVERSITY * university[i]
        + B_PUBLIC_BUILDING * public_building[i]
        + B_PRIVATE_BUILDING * private_building[i]
        + B_PUBLIC_PARK * public_park[i]
        + B_HOSPITAL * hospital[i]
        + B_TECHNICAL_INSTITUTION * technical_institution[i]
        + B_SCIENCE_PARK * science_park[i]
        + B_RECREATIONAL_PARK * recreational_park[i]
        + B_DOWNTOWN_RECYCLING * downtown_recycling[i]
        + B_CONV_CENTER * conv_center[i]
        + B_INDUSTRIAL_BUSINESS_CENTER * industrial_business_center[i]
    )


In [ ]:
av = {i: 1 for i in range(1, 36)}

logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'rvm_logit_distance_location'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

Second model - distance only. Testing distance

In [ ]:
B_DIST = Beta('B_DIST', 0, None, None, 0)

V = {}
for i in range(1, 36):
    V[i] = B_DIST * dist[i]

av = {i: 1 for i in range(1, 36)}

logprob = models.loglogit(V, av, choice_code)
biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'rvm_model_2'

results = biogeme.estimate()
print(results.short_summary())
print(results.get_estimated_parameters())

Third model - only chosen machines. As mentioned before, choice code has only 12 unique values, meaning only 12 different machines were chosen. Good log, AIC and BIC but "Warning: identification issue" .

In [ ]:
chosen_alternatives = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14]

V = {}
for i in chosen_alternatives:
    V[i] = (
        B_DIST * dist[i]
        + B_INDOOR * indoor[i]
        + B_UNIVERSITY * university[i]
        + B_PUBLIC_BUILDING * public_building[i]
        + B_PRIVATE_BUILDING * private_building[i]
        + B_PUBLIC_PARK * public_park[i]
        + B_HOSPITAL * hospital[i]
        + B_TECHNICAL_INSTITUTION * technical_institution[i]
        + B_SCIENCE_PARK * science_park[i]
        + B_RECREATIONAL_PARK * recreational_park[i]
        + B_DOWNTOWN_RECYCLING * downtown_recycling[i]
        + B_CONV_CENTER * conv_center[i]
        + B_INDUSTRIAL_BUSINESS_CENTER * industrial_business_center[i]
    )

av = {i: 1 for i in chosen_alternatives}

logprob = models.loglogit(V, av, choice_code)
biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'rvm_model_3'

results = biogeme.estimate()
print(results.short_summary())
print(results.get_estimated_parameters())

In [ ]:
location_vars = [
    'indoor',
    'university',
    'public_building',
    'private_building',
    'public_park',
    'hospital',
    'technical_institution',
    'science_park',
    'recreational_park',
    'downtown_recycling',
    'conv_center',
    'industrial_business_center',
]

for var in location_vars:
    chosen_count = 0
    machine_count = 0

    for i in range(1, 36):
        col = f'{var}_{i}'
        if col in df.columns:
            machine_count += int(df[col].max() == 1)
            chosen_count += ((df['choice_code'] == i) & (df[col] == 1)).sum()

    print(var, 'machines:', machine_count, 'chosen observations:', chosen_count)

Fourth model - distance and distance_indoor

In [ ]:
B_DIST = Beta('B_DIST', 0, None, None, 0)
B_DIST_INDOOR = Beta('B_DIST_INDOOR', 0, None, None, 0)

V = {}

for i in range(1, 36):
    V[i] = (
        B_DIST * dist[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
    )

av = {i: 1 for i in range(1, 36)}

logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'rvm_4model_distance_indoor'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

model 5: grouped locations and indoor on distance

In [ ]:
shopping_center = {i: Variable(f'shopping_center_{i}') for i in range(1, 36)}

# Grouped location variables
building = {}
campus = {}
lowuse = {}
commercial = {}

for i in range(1, 36):


    building[i] = (
        public_building[i]
        + private_building[i]
    )

    campus[i] = (
        university[i]
        + technical_institution[i]
        + science_park[i]
    )

    lowuse[i] = (
        public_park[i]
        + hospital[i]
        + recreational_park[i]
        + downtown_recycling[i]
        + industrial_business_center[i]
    )

    commercial[i] = (
        shopping_center[i]
        + conv_center[i]
    )

B_DIST = Beta('B_DIST', 0, None, None, 0)
B_DIST_INDOOR = Beta('B_DIST_INDOOR', 0, None, None, 0)

B_BUILDING = Beta('B_BUILDING', 0, None, None, 0)
B_CAMPUS = Beta('B_CAMPUS', 0, None, None, 0)
B_LOWUSE = Beta('B_LOWUSE', 0, None, None, 0)


V = {}

for i in range(1, 36):
    V[i] = (
        B_DIST * dist[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
        + B_BUILDING * building[i]
        + B_CAMPUS * campus[i]
        + B_LOWUSE * lowuse[i]
    )



av = {i: 1 for i in range(1, 36)}



logprob = models.loglogit(V, av, choice_code)



biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'rvm_grouped_location_indoor_distance_model5'
results = biogeme.estimate()
print(results.short_summary())
print(results.get_estimated_parameters())

In [ ]:
B_DIST = Beta('B_DIST', 0, None, None, 0)
B_DIST_SOCIO = Beta('B_DIST_SOCIO', 0, None, None, 0)

socioeconomic_level = Variable('socioeconomic_level')

V = {}

for i in range(1, 36):
    V[i] = (
        B_DIST * dist[i]
        + B_DIST_SOCIO * dist[i] * socioeconomic_level
    )

av = {i: 1 for i in range(1, 36)}


logprob = models.loglogit(V, av, choice_code)


biogeme = bio.BIOGEME(database, logprob)
biogeme.modelName = 'rvm_soclevel_dist'
results = biogeme.estimate()
print(results.short_summary())
print(results.get_estimated_parameters())

model 6 - using age and female and socio level

In [ ]:
Female = Variable('Female')
Age_10 = Variable('Age_10')
socio_area_2 = {i: Variable(f'socioeconomic_level_2_{i}') for i in range(1, 36)}
socio_area_3 = {i: Variable(f'socioeconomic_level_3_{i}') for i in range(1, 36)}
socio_area_4 = {i: Variable(f'socioeconomic_level_4_{i}') for i in range(1, 36)}
socio_area_5 = {i: Variable(f'socioeconomic_level_5_{i}') for i in range(1, 36)}
socio_area_6 = {i: Variable(f'socioeconomic_level_6_{i}') for i in range(1, 36)}

In [ ]:
B_SOCIO_AREA_2 = Beta('B_SOCIO_AREA_2', 0, None, None, 0)
B_SOCIO_AREA_3 = Beta('B_SOCIO_AREA_3', 0, None, None, 0)
B_SOCIO_AREA_4 = Beta('B_SOCIO_AREA_4', 0, None, None, 0)
B_SOCIO_AREA_5 = Beta('B_SOCIO_AREA_5', 0, None, None, 0)
B_SOCIO_AREA_6 = Beta('B_SOCIO_AREA_6', 0, None, None, 0)
B_DIST = Beta('B_DIST', 0, None, None, 0)
B_DIST_INDOOR = Beta('B_DIST_INDOOR', 0, None, None, 0)

B_DIST_AGE = Beta('B_DIST_AGE', 0, None, None, 0)
B_DIST_FEMALE = Beta('B_DIST_FEMALE', 0, None, None, 0)

B_INDOOR_DIST_AGE = Beta('B_INDOOR_DIST_AGE', 0, None, None, 0)
B_INDOOR_DIST_FEMALE = Beta('B_INDOOR_DIST_FEMALE', 0, None, None, 0)

B_BUILDING = Beta('B_BUILDING', 0, None, None, 0)
B_CAMPUS = Beta('B_CAMPUS', 0, None, None, 0)
B_LOWUSE = Beta('B_LOWUSE', 0, None, None, 0)

In [ ]:
V = {}

for i in range(1, 36):
    V[i] = (
        B_DIST * dist[i]
       # + B_DIST_INDOOR * dist[i] * indoor[i]

       # + B_DIST_AGE * dist[i] * Age
       # + B_DIST_FEMALE * dist[i] * Female

        + B_INDOOR_DIST_AGE * dist[i] * indoor[i] * Age_10
        + B_INDOOR_DIST_FEMALE * dist[i] * indoor[i] * Female

        + B_BUILDING * building[i]
        #+ B_CAMPUS * campus[i]
        + B_LOWUSE * lowuse[i]
    )
av = {i: 1 for i in range(1, 36)}

logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_age_female_model8_adjusted'
results = biogeme.estimate()
print(results.short_summary())
print(results.get_estimated_parameters())

In [ ]:
print(df['id'].value_counts().describe())


In [ ]:
df["Age_10"]

In [ ]:
av = {i: 1 for i in machines_to_keep}

logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model9_tutor_advice'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

In [ ]:
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model10_tutor_advice'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())